In [ ]:
# Runs all of EUBAR until the regression step
# Used to show which probes are selected for each motif position
# and how they are split across alleles
# This is a simplified version of the main function

from utils import (
    read_intensities, 
    read_kmer_positions,
    get_mutation_sequence, 
    get_kmer_variants,
    match_kmers_to_wildcards, 
    select_random_probes,
    project_kmers_to_random_probes
)


intensities = read_intensities("./data/ENCFF003ZRP_ENCFF274YGF.ChIPBM.tf.txt")
kmers = read_kmer_positions("./data/ENCFF274YGF_K562_DNase-seq_Crawford_GRCh38.ChIPBM.array.txt")
snv_list = "chr6:41071106:C>T,chr2:106194416:C>G"
genome =  "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
dhs = 1


def main(
    snv_list,
    kmers,
    kmer_size,
    intensities,
    genome,
    num_random=500,
    dhs=False,
    mode="neg-binomial"
):
    # Collect all used regions from the kmer mapping
    used_regions = set(kmers.keys())

    for snv in snv_list.split(","):
        chrom, pos, refalt = snv.split(":")
        ref, alt = refalt.split(">")
        pos = int(pos)

        # Get the motif sequence centered at the SNP
        motif = get_mutation_sequence(chrom, pos, ref, genome, kmer_size)

        chrom, pos, refalt = snv.split(":")
        ref, alt = refalt.split(">")
        pos = int(pos)

        # Step 1: get motif and variants
        motif_seed = get_mutation_sequence(chrom, pos, ref, genome, kmer_size)
        print(motif_seed)
        kmer_list, wildcard_variants, snp_indices = get_kmer_variants(motif_seed, kmer_size)
        print(kmer_list, wildcard_variants, snp_indices)

        # Step 2: match wildcards to genome kmers
        comp_alleles, matched = match_kmers_to_wildcards(
            kmers, wildcard_variants, snp_indices, motif_seed
        )

        # Step 3: select and split random probes ONCE per SNP
        rand_seed = hash(snv) % (2**32)
        rand_probes = select_random_probes(kmers, used_regions, num_random=num_random, seed=rand_seed)
        split_random = project_kmers_to_random_probes(rand_probes, wildcard_variants, kmer_list, kmers)


        # Step 4: run regression for each motif position
        regression_results = {}

        for motif_pos in comp_alleles:
            for snp_index in comp_alleles[motif_pos]:
                ref_allele = comp_alleles[motif_pos][snp_index]

                matched_block = {motif_pos: {snp_index: matched[motif_pos]}}
                random_block = {motif_pos: {snp_index: split_random[motif_pos]}}
                comp_block   = {motif_pos: {snp_index: ref_allele}}
    return matched_block, random_block, comp_block, matched


matched_block, random_block, comp_block, matched = main(
    snv_list="chr6:41071106:C>T",
    kmers=kmers,
    kmer_size=8,
    intensities=intensities,
    genome=genome,
    dhs = dhs
)


/home/aki/miniconda3/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


GGGCAAACCAGGTAA
['GGGCAAAC', 'GGCAAACC', 'GCAAACCA', 'CAAACCAG', 'AAACCAGG', 'AACCAGGT', 'ACCAGGTA', 'CCAGGTAA'] {0: 'GGGCAAA.', 1: 'GGCAAA.C', 2: 'GCAAA.CA', 3: 'CAAA.CAG', 4: 'AAA.CAGG', 5: 'AA.CAGGT', 6: 'A.CAGGTA', 7: '.CAGGTAA'} {0: 7, 1: 6, 2: 5, 3: 4, 4: 3, 5: 2, 6: 1, 7: 0}


In [2]:
# This function was used to return the k-mers that match a specific region and position.
# Old, not really useful anymore.

def find_kmers_by_region_and_position(kmers, region, position=None):
    matches = []
    for kmer, region_dict in kmers.items():
        if region in region_dict:
            offset = region_dict[region]
            if position is None or offset == position:
                matches.append((kmer, offset))
    return matches

# Example usage:
target_region = "chr7:1138260-1138440"
target_position = 11

# With position filter
results = find_kmers_by_region_and_position(kmers, target_region, target_position)

# Or without position filter (returns all k-mers that match the region)
# results = find_kmers_by_region_and_position(kmers, target_region)

# Display results
for kmer, offset in results:
    print(f"K-mer: {kmer}, Offset: {offset}")

K-mer: CCAGGTAA, Offset: 11


In [3]:
# Showcase that the perl tool does not function properly
# This is the AFF, final position, so .CAGGTAA
# In this case, we are looking for all the probes that match the above, where . is a wild card
# We expect it to only return CCAGGTAA, GCAGGTAA, TCAGGTAA, ACAGGTAA
# It returns more than that, which is incorrect

from collections import Counter
import re

def find_kmers_by_region_and_position(kmers, region, position=None):
    matches = []
    for kmer, region_dict in kmers.items():
        if region in region_dict:
            offset = region_dict[region]
            if position is None or offset == position:
                matches.append((kmer, offset))
    return matches

# Step 1: Parse the file
def parse_perl_dump_file(filepath):
    region_offsets = []
    with open(filepath) as f:
        current_region = None
        for line in f:
            if line.startswith("Matched probe region:"):
                match = re.search(r"Matched probe region:\s*(\S+)", line)
                if match:
                    current_region = match.group(1)
            elif "Matched k-mer offset:" in line:
                match = re.search(r"Matched k-mer offset:\s*(\d+)", line)
                if match and current_region:
                    offset = int(match.group(1))
                    region_offsets.append((current_region, offset))
    return region_offsets

# Step 2: Tally k-mers
def tally_kmers(kmers, region_offsets):
    kmer_counts = Counter()
    for region, offset in region_offsets:
        found = find_kmers_by_region_and_position(kmers, region, offset)
        for kmer, _ in found:
            kmer_counts[kmer] += 1
    return kmer_counts

# Run it
region_offset_list = parse_perl_dump_file("tests/last_pos_motif_perl.txt")
kmer_frequencies = tally_kmers(kmers, region_offset_list)

# Print top k-mers
for kmer, count in kmer_frequencies.most_common():
    print(f"{kmer}: {count}")


CCAGGTAA: 212
GCAGGTAA: 200
TCAGGTAA: 176
ACAGGTAA: 168
GGCTCCTG: 2
GTGCCAGG: 2
GAGCATCA: 2
ATAAACTG: 2
GGAGGGTT: 2
CCCTGAGG: 2
GCCCCGCC: 2
GGCCCTGC: 2
TCCTGCCT: 2
AAATGCAG: 2
GAGCCGGG: 2
CCCCTTTG: 2
CGGGGTCG: 2
AAACTTGT: 2
TGGCCACT: 2
GACTGGCT: 2
GGCGCGGC: 1
AAAAACCA: 1
CTCTCTTT: 1
AGAAAACA: 1
AACACTGG: 1
AGCCTCAA: 1
GAAATTGT: 1
TCTGCTCC: 1
CCTGTTAC: 1
GGTTGCAC: 1
TCTGGGCT: 1
AGGCTCGC: 1
TCTCTCTC: 1
ACCTAGCT: 1
GTCACGTG: 1
GGGCGTTG: 1
TCATTCAT: 1
CTTCTAGA: 1
CCGTAGGT: 1
TCGGCCTG: 1
ACTCAAAC: 1
CGTGAATC: 1
CAATGACG: 1
CTTTACCA: 1
CCCTCCCT: 1
AGATGGCG: 1
CATAGAGG: 1
GCAGGCCC: 1
AGCTGCCA: 1
CTTCTCGT: 1
TCTTTTGC: 1
CCCCATCC: 1
TAGGTGGG: 1
GCTCATGA: 1
GACTTGGA: 1
AGGCCTGG: 1
AGCTTGCC: 1
GGTGAGGT: 1
ACAAAGTC: 1
AGGGCTTT: 1
GGGCGGCG: 1
GCGGGCTC: 1
TGCTGTAT: 1
AAATGTTC: 1
GGAGAGTT: 1
CAATCATC: 1
GTTGGAAC: 1
CTTCTCTG: 1
AGCTCTAG: 1
TAGAGCTG: 1
TCTGTAAT: 1
TTACCACT: 1
CCTCTAGC: 1
GACCTGAG: 1
GCACATTT: 1
TGAATATG: 1
TTCCGGCA: 1
CAGGGTCC: 1
CGCCATCT: 1
TCTCACCC: 1
GCTCCAAG: 1
ATGGACTC: 1
GACTCTCG

In [ ]:
# This is used mainly for debugging
# The outputs : all_matched_blocks and all_random_blocks contain all the matches for every position in the sliding window

def main(
    snv_list,
    kmers,
    kmer_size,
    intensities,
    genome,
    num_random=500,
    dhs=False,
    mode="neg-binomial"
):
    used_regions = set(kmers.keys())
    all_matched_blocks = {}
    all_random_blocks = {}
    all_comp_blocks = {}

    for snv in snv_list.split(","):
        chrom, pos, refalt = snv.split(":")
        ref, alt = refalt.split(">")
        pos = int(pos)

        motif_seed = get_mutation_sequence(chrom, pos, ref, genome, kmer_size)
        kmer_list, wildcard_variants, snp_indices = get_kmer_variants(motif_seed, kmer_size)

        comp_alleles, matched = match_kmers_to_wildcards(
            kmers, wildcard_variants, snp_indices, motif_seed
        )

        rand_seed = hash(snv) % (2**32)
        rand_probes = select_random_probes(kmers, used_regions, num_random=num_random, seed=rand_seed)
        split_random = project_kmers_to_random_probes(rand_probes, wildcard_variants, kmer_list, kmers)

        for motif_pos in comp_alleles:
            for snp_index in comp_alleles[motif_pos]:
                ref_allele = comp_alleles[motif_pos][snp_index]

                all_matched_blocks[(snv, motif_pos, snp_index)] = {motif_pos: {snp_index: matched[motif_pos]}}
                all_random_blocks[(snv, motif_pos, snp_index)] = {motif_pos: {snp_index: split_random[motif_pos]}}
                all_comp_blocks[(snv, motif_pos, snp_index)]   = {motif_pos: {snp_index: ref_allele}}

    return all_matched_blocks, all_random_blocks, all_comp_blocks, rand_probes, split_random


all_matched_blocks, all_random_block, all_comp_block, rand_probes, split_random = main(
    snv_list="chr6:41071106:C>T",
    kmers=kmers,
    kmer_size=8,
    intensities=intensities,
    genome=genome,
    num_random = 500,
    dhs = dhs
)

all_matched_blocks

In [25]:

all_random_block

{('chr6:41071106:C>T',
  0,
  7): {0: {7: defaultdict(dict,
               {'A': {'chr13:93361680-93362080': 140,
                 'chr8:125312720-125312920': 3,
                 'chr12:54379720-54379940': 3,
                 'chr9:131382300-131382560': 116},
                'T': {'chr11:34588140-34588300': 117}})}},
 ('chr6:41071106:C>T',
  1,
  6): {1: {6: defaultdict(dict,
               {'A': {'chr2:230219940-230220160': 26},
                'C': {'chr1:27538886-27539055': 19},
                'G': {'chr13:108002640-108003120': 249,
                 'chr19:56567360-56567560': 25},
                'T': {'chr2:46564800-46565020': 202}})}},
 ('chr6:41071106:C>T',
  2,
  5): {2: {5: defaultdict(dict,
               {'A': {'chr2:230219940-230220160': 27,
                 'chr6:145735940-145736160': 175},
                'C': {'chr1:178093180-178093460': 78,
                 'chr14:101584200-101584400': 104,
                 'chr1:211711480-211711802': 49,
                 'chr1:27538886

In [6]:
print(find_kmers_by_region_and_position(kmers, 'chr14:96634380-96634700', 39))
print(find_kmers_by_region_and_position(kmers, 'chr6:149140140-149140320', 156))
print(find_kmers_by_region_and_position(kmers, 'chr2:219253100-219253480', 316))
print(find_kmers_by_region_and_position(kmers, 'chr7:74051780-74051960', 56))
print(find_kmers_by_region_and_position(kmers, 'chr2:63051100-63051600', 214))
print(find_kmers_by_region_and_position(kmers, 'chr15:51829820-51830200', 13))



[('ACAGGTAA', 39)]
[('CCAGGTAA', 156)]
[('CCAGGTAA', 316)]
[('GCAGGTAA', 56)]
[('GCAGGTAA', 214)]
[('TCAGGTAA', 13)]


In [24]:
# Function that summarizes number the matched probes
# The order is always : SNP, motif_pos, snp_index

def summarize_matched_probes(matched_block):
    summary = {}

    for key, value in matched_block.items():
        snp_id, motif_pos, snp_index = key
        summary[key] = {}

        # CAREFUL: the key nesting is [motif_pos][snp_index] not [snp_index][motif_pos]
        allele_dict = value[motif_pos][snp_index]

        for allele, regions in allele_dict.items():
            summary[key][allele] = len(regions)

    return summary


summary = summarize_matched_probes(all_random_block)

for key, allele_counts in summary.items():
    print(f"\nSNP: {key}")
    for allele, count in allele_counts.items():
        print(f"  Allele {allele}: {count} probes")



SNP: ('chr6:41071106:C>T', 0, 7)
  Allele A: 4 probes
  Allele T: 1 probes

SNP: ('chr6:41071106:C>T', 1, 6)
  Allele A: 1 probes
  Allele C: 1 probes
  Allele G: 2 probes
  Allele T: 1 probes

SNP: ('chr6:41071106:C>T', 2, 5)
  Allele A: 2 probes
  Allele C: 4 probes
  Allele G: 4 probes
  Allele T: 1 probes

SNP: ('chr6:41071106:C>T', 3, 4)
  Allele A: 2 probes
  Allele C: 5 probes
  Allele G: 2 probes
  Allele T: 1 probes

SNP: ('chr6:41071106:C>T', 4, 3)
  Allele A: 5 probes
  Allele C: 3 probes
  Allele G: 2 probes

SNP: ('chr6:41071106:C>T', 5, 2)
  Allele C: 1 probes
  Allele G: 2 probes
  Allele T: 1 probes

SNP: ('chr6:41071106:C>T', 6, 1)
  Allele G: 2 probes
  Allele T: 1 probes

SNP: ('chr6:41071106:C>T', 7, 0)
  Allele C: 1 probes
  Allele G: 1 probes
  Allele T: 1 probes


In [8]:
# In the last position of chr6:41071106:C>T, how many probes match per allele?

# List of k-mers you want to summarize
target_kmers = ["ACAGGTAA", "TCAGGTAA", "CCAGGTAA", "GCAGGTAA"]

# Dictionary to hold counts
kmer_probe_counts = {}

# Loop through target k-mers and count how many probe regions are associated
for kmer in target_kmers:
    if kmer in kmers:
        kmer_probe_counts[kmer] = len(kmers[kmer])
    else:
        kmer_probe_counts[kmer] = 0  # No matches found

# Print summary
for kmer, count in kmer_probe_counts.items():
    print(f"{kmer}: {count} probes")
    
subset_kmers = {kmer: kmers[kmer] for kmer in target_kmers if kmer in kmers}


ACAGGTAA: 175 probes
TCAGGTAA: 179 probes
CCAGGTAA: 214 probes
GCAGGTAA: 207 probes


In [9]:
# How many probes are there in kmers?
def count_total_unique_probes(kmers):
    all_probes = set()
    for region_dict in kmers.values():
        all_probes.update(region_dict.keys())
    return len(all_probes)

total_probes = count_total_unique_probes(kmers)
print(f"Total unique probe regions: {total_probes}")

# Maybe use this function to automatically set rand kmers? A percentage? Sample 10% of the total?

Total unique probe regions: 202980


In [10]:
from collections import defaultdict

def summarize_matched_block(block):
    summary = defaultdict(int)

    for motif_pos in block:
        for snp_index in block[motif_pos]:
            allele_dict = block[motif_pos][snp_index]
            for allele, region_dict in allele_dict.items():
                summary[allele] += len(region_dict)

    return dict(summary)

summary = summarize_matched_block(matched_block)
print(summary)


{'A': 175, 'G': 207, 'T': 179, 'C': 214}


In [11]:
# List of kmers to inspect
target_kmers = ["ACAGGTAA", "TCAGGTAA", "CCAGGTAA", "GCAGGTAA"]

# Summarize how many regions each has
kmer_region_counts = {kmer: len(kmers.get(kmer, {})) for kmer in target_kmers}

print(kmer_region_counts)


{'ACAGGTAA': 175, 'TCAGGTAA': 179, 'CCAGGTAA': 214, 'GCAGGTAA': 207}


In [ ]:
matched_regions = set()
for motif_pos in matched_block:
    for snp_index in matched_block[motif_pos]:
        for allele in matched_block[motif_pos][snp_index]:
            matched_regions.update(matched_block[motif_pos][snp_index][allele].keys())
matched_regions

In [21]:
# Sanity check
total_matched = {'ACAGGTAA': 175, 'TCAGGTAA': 179, 'CCAGGTAA': 214, 'GCAGGTAA': 207}
expected_matched_regions = set()

# Build expected matched region set
for kmer in total_matched:
    expected_matched_regions.update(kmers[kmer].keys())

# Then check overlap with random probes
random_probes = select_random_probes(kmers, expected_matched_regions, num_random=500)

# Count accidental matches
accidental_matches = [
    region for region in random_probes if region in expected_matched_regions
]

print(f"Total matched regions: {len(expected_matched_regions)}")
print(f"Random probes selected: {len(random_probes)}")
print(f"Accidental overlaps: {len(accidental_matches)}")


Total matched regions: 773
Random probes selected: 499
Accidental overlaps: 0


In [29]:
def find_kmers_for_region(target_region, target_offset, kmers_dict):
    """
    Given a region and offset (from a random probe), find all kmers in the kmers_dict
    that include that region with that offset.

    Args:
        target_region (str): e.g. "chr1:1000-1160"
        target_offset (int): e.g. 40
        kmers_dict (dict): {kmer: {region: offset}}

    Returns:
        list of kmers (str) that match the region + offset
    """
    matches = []
    for kmer, region_map in kmers_dict.items():
        if region_map.get(target_region) == target_offset:
            matches.append(kmer)
    return matches


# Say we had a random probe selected like:
region = "chr13:108002640-108003120"
offset = 280

matching_kmers = find_kmers_for_region(region, offset, kmers)
print(f"K-mers that matched region {region} at offset {offset}:")
print(matching_kmers)


K-mers that matched region chr13:108002640-108003120 at offset 280:
['TCAGGTAA']


In [ ]:
def find_random_regions_with_target_kmers(kmer_targets, rand_block, kmers):
    target_regions = set()
    for kmer in kmer_targets:
        if kmer in kmers:
            target_regions.update(kmers[kmer].keys())

    hits = []
    for key, subdict in rand_block.items():
        motif_pos, snp_index = key[1], key[2]
        allele_dict = subdict[motif_pos][snp_index]
        for allele, region_dict in allele_dict.items():
            for region in region_dict:
                if region in target_regions:
                    hits.append((region, allele))

    return hits


kmer_targets = ["ACAGGTAA", "TCAGGTAA", "CCAGGTAA", "GCAGGTAA"]
random_hits = find_random_regions_with_target_kmers(kmer_targets, all_random_block, kmers)

print(f"Total overlapping randoms: {len(random_hits)}")
for region, allele in random_hits[:10]:
    print(f"{region} (allele {allele})")


Total overlapping randoms: 5
chr13:108002640-108003120 (allele G)
chr11:68632920-68633080 (allele G)
chr1:18526892-18527544 (allele C)
chr11:68632920-68633080 (allele G)
chr13:108002640-108003120 (allele T)


In [2]:
genome =  "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
def get_sequence_from_fasta(chrom, start, end, genome_fasta_path):
    from pyfaidx import Fasta
    """
    Extracts the DNA sequence from a given genomic region.

    Args:
        chrom (str): Chromosome name (e.g., "chr6")
        start (int): 1-based start position (inclusive)
        end (int): 1-based end position (inclusive)
        genome_fasta_path (str): Path to the genome FASTA file

    Returns:
        str: The DNA sequence from the region
    """
    genome = Fasta(genome_fasta_path)
    sequence = genome[chrom][start - 1:end].seq.upper()
    return sequence

seq = get_sequence_from_fasta("chr6", 41071098, 41071113, genome)
print(seq)


AGGGCAAACCAGGTAA
